In [4]:
!pip install dask_geopandas

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 17.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 6/8 [dask]  WARNING: The script dask is installed in '/home/jovyan/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [dask_geopandas]m [dask_geopandas]

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
import warnings
import logging
import time
from datetime import datetime
import psutil
import os
import sys

# Add parent directory to Python path to import project modules
sys.path.insert(0, os.path.abspath('..'))

In [2]:
from process.append import check_core_criteria, append_enriched_features, get_all_enriched_paths, get_enriched_features

In [3]:
import geopandas as gpd
import dask_geopandas

In [5]:
enriched_path = "../../_User-Persistent-Storage_CephBlock_/V2.1"
reference_path = "a_Reference.gdb"
california_boundary_layer_name = "California"
output_append_path = "../../_User-Persistent-Storage_CephBlock_/V2.1_OUT"
start_year = 1950
end_year = 2025
# Input must be from ["point", "line", "polygon", "all"].
geom_type = "polygon" 

In [6]:

enriched_layers = get_all_enriched_paths(enriched_path)


# if only a specific geom_type needs to be processed
if geom_type == "point":
    enriched_layers = {'point': enriched_layers['point'], 
                'line': [],
                'polygon': []}
elif geom_type == "line":
    enriched_layers = {'point': [], 
                'line': enriched_layers['line'],
                'polygon': []}
elif geom_type == "polygon":
    enriched_layers = {'point': [], 
                'line': [],
                'polygon': enriched_layers['polygon']}
elif geom_type == "all":
    pass
else:
    raise ValueError('Input must be from ["point", "line", "polygon", "all"].')



In [9]:
# read enriched geodatabase to geopandas gdf and concat together
enriched_polygons, enriched_lines, enriched_points = get_enriched_features(enriched_layers)


2026-05-26 21:56:01,788 INFO  [process.append_polygon]  --------------------------------------------------------------------------------
2026-05-26 21:56:01,789 INFO  [process.append_polygon]  Concatenate all polygon records
2026-05-26 21:56:01,790 INFO  [process.append_polygon]  Load GeoDataFrame from the layer 'Timber_Industry_Spatial_20260504' in '../../_User-Persistent-Storage_CephBlock_/V2.1/Timber_Industry_Spatial_1950_2025.gdb' 
/usr/local/lib/python3.10/dist-packages/pyogrio/raw.py:198: RuntimeWarning: driver OpenFileGDB does not support open option DRIVER
  return ogr_read(
2026-05-26 21:56:02,078 INFO  [process.append_polygon]  Load GeoDataFrame from the layer 'USFS_Region04_enriched_20260511' in '../../_User-Persistent-Storage_CephBlock_/V2.1/USFS_1950_2025.gdb' 
/usr/local/lib/python3.10/dist-packages/pyogrio/raw.py:198: RuntimeWarning: driver OpenFileGDB does not support open option DRIVER
  return ogr_read(
2026-05-26 21:56:03,006 INFO  [process.append_polygon]  Load GeoD

In [33]:
summary_cols = ['AGENCY', 'ACTIVITY_STATUS', 'ADMINISTERING_ORG', 'PRIMARY_OWNERSHIP_GROUP', 'REGION', 'ACTIVITY_CAT', 'BROAD_VEGETATION_TYPE']

In [40]:
# display unique value of core columns
# Set COUNTS_TO_MAS == 'NO' for None rows
for c in summary_cols:
    print(enriched_polygons[c].unique())
    if None in enriched_polygons[c].unique():
        enriched_polygons.loc[enriched_polygons[c].isna(), 'COUNTS_TO_MAS'] = 'NO'

['TIMBER' 'USDA' 'CNRA' 'OTHER' 'DOI' 'NPS']
['COMPLETE' 'ACTIVE' 'PLANNED' 'CANCELLED' 'PROPOSED' 'OUTYEAR' None]
['TIMBER' 'USFS' 'SDRC' 'RMC' 'BOF' 'SCC' 'SMMC' 'DOC' 'NRCS' 'SLC'
 'PARKS' 'WCB' 'SNC' 'CALFIRE' 'CDFW' 'TAHOE' 'CCC' None 'BLM' 'FWS' 'NPS'
 'DOD' 'BIA']
['FEDERAL' 'PRIVATE_NON-INDUSTRY' 'PRIVATE_INDUSTRY' 'STATE' 'NGO' 'LOCAL'
 'TRIBAL']
['SIERRA_NEVADA' 'NORTH_COAST' 'SOUTHERN_CA' 'CENTRAL_COAST' None]
['MECH_HFR' 'PRESCRB_FIRE' 'TIMB_HARV' 'TREE_PLNTING' 'NOT_DEFINED'
 'SANI_SALVG' 'GRAZING' 'LAND_PROTEC']
['FOREST' 'SPARSE' 'GRASS_HERB' 'WATER' 'SHRB_CHAP' 'URBAN' 'AGRICULTURE'
 'WETLAND' None]


In [18]:
# read california boundary for cliping
california_boundary = gpd.read_file(reference_path, 
                                driver='OpenFileGDB', 
                                layer=california_boundary_layer_name)

# grab timber non spatial path again
timber_nonspatial_path = None
timber_nonspatial = None
for p in enriched_layers['point']:
    if 'Timber_Nonspatial' in p['gdb_path']:
        timber_nonspatial_path = p
        break
if timber_nonspatial_path:
    timber_nonspatial = gpd.read_file(timber_nonspatial_path['gdb_path'], 
                                driver='OpenFileGDB', 
                                layer=timber_nonspatial_path['layer_name'])


/usr/local/lib/python3.10/dist-packages/pyogrio/raw.py:198: RuntimeWarning: driver OpenFileGDB does not support open option DRIVER
  return ogr_read(


In [ ]:



# use dask geopandas for multithread clipping
for df, lyr_name in zip([enriched_polygons,enriched_lines,enriched_points], ["appended_poly","appended_line","appended_point"]):
    # init dask gdf for multithread clipping
    ddf = dask_geopandas.from_geopandas(df, npartitions=16)
    # clip to california bounds
    append_clipped = ddf.sjoin(california_boundary, how='inner', predicate='intersects').compute()
    # drop unwanted artifact columns from California boundary df
    append_clipped = append_clipped.drop(['index_right', 'Shape_Area', 'Shape_Length'], axis=1)
    
    # industry nonspatial is by design out of california bounds and got clipped, manually concat it back
    if lyr_name == 'appended_point':
        append_clipped = pd.concat([append_clipped, timber_nonspatial], ignore_index=True)

    append_clipped['CORE_CRITERIA'] = append_clipped.apply(check_core_criteria, axis=1)
    save_gdf_to_gdb(append_clipped, output_append_path, lyr_name)